# DPE weighting study — one end-to-end Kaggle / Colab notebook

**No existing H2 ZIP is required.** Select a **GPU runtime**, enable Internet on Kaggle, and choose **Run All**. The default runs standard Waterbirds seeds **0, 1 and 2**: dataset setup, a 100-epoch ERM backbone per seed, feature extraction, 15 DPE heads, every weighting rule, test evaluation and a combined report/download.

If an original H2 ZIP is attached, matching seeds reuse its heads/predictions automatically. Other seeds are trained automatically. This saves the work already completed for seed 0. `MODE="cached"` remains available for a fast ZIP-only comparison.

The notebook clones the experiment branch when needed and installs missing packages. Training takes real GPU time; automation removes the manual handoffs. Backbone checkpoints are saved at epoch boundaries. Rerun after an interruption to resume saved epochs and reuse completed stages. **Keep WORK_DIR persistent** to survive VM deletion: for Colab you may set it to an already mounted Drive directory; for Kaggle save outputs/checkpoints before the session is discarded. The compact results ZIP excludes large backbone/resume checkpoints and raw features.

All weights fit on B; C selects configurations before new test predictions are generated. The fixed rule grid is unchanged across seeds. See the [method instructions](../src/methods/dpe_head_reweighting/README.md#one-end-to-end-weighting-study).

**Recover a failed Kaggle run:** attach that failed version’s saved notebook outputs as an input dataset, then restart the kernel and Run All. Completed heads and matching raw feature caches are discovered automatically, even when weighting failed before test inference. Existing training is preserved.


In [ ]:
# Normally no edits are needed. Default study settings live in configs/dpe_head_reweighting/study.yaml.
MODE = "auto"       # "auto": train missing seeds; "cached": use existing H2 ZIPs only.
SEEDS = None        # Default [0, 1, 2]; use [1, 2] to run only the remaining seeds.
TRACK = None        # Default "standard"; alternatively "enhanced".
SOURCE = None       # Optional original H2 ZIP/directory, or list of ZIPs/directories.
DATA_DIR = None     # Automatically discover attached Waterbirds, or download it once.
DEVICE = "auto"    # Prefer GPU when available; CPU is supported but training is slow.
WORKERS = None      # Default 2; use 0 if the runtime cannot start loader workers.

# Optional overrides. WORK_DIR should be persistent if you want cross-session resume.
REPO_PATH = None
WORK_DIR = None
SEARCH_PATHS = None
REPO_URL = "https://github.com/chautanphat/waterbirds-shortcut-mitigation.git"
REPO_REF = "experiment/dpe-multiple-weighting-rules"


In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

working = Path.cwd().resolve()
PLATFORM = "kaggle" if Path("/kaggle/working").is_dir() else "colab" if Path("/content").is_dir() else "local"
candidates = [working, *working.parents, working / "waterbirds-shortcut-mitigation"]
REPO = Path(REPO_PATH).expanduser().resolve() if REPO_PATH else next(
    (p for p in candidates if (p / "src/methods/dpe_head_reweighting/recovery.py").is_file()), None)
if WORK_DIR is None:
    base = Path("/kaggle/working") if PLATFORM == "kaggle" else Path("/content") if PLATFORM == "colab" else (REPO / "outputs" if REPO else working)
    WORK_DIR = base / "dpe_weighting_rules_session"
WORK_DIR = Path(WORK_DIR).expanduser().resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)
if REPO is None:
    REPO = WORK_DIR / "repository-solver-recovery-v1"
    if not REPO.exists():
        try:
            subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", REPO_REF,
                            REPO_URL, str(REPO)], check=True)
        except subprocess.CalledProcessError as error:
            raise RuntimeError("Repository download failed. On Kaggle, enable Internet; alternatively supply REPO_PATH to an updated checkout.") from error
if not (REPO / "src/methods/dpe_head_reweighting/recovery.py").is_file():
    raise FileNotFoundError("REPO_PATH must contain the updated multi-rule implementation. Use the experiment/dpe-multiple-weighting-rules branch.")
missing = [package for module, package in [
    ("numpy", "numpy>=1.26,<3"), ("scipy", "scipy>=1.11,<2"),
    ("torch", "torch>=2.6,<3"), ("torchvision", "torchvision>=0.21,<1"),
    ("yaml", "PyYAML>=6,<7"), ("pandas", "pandas>=2.2,<3"),
    ("PIL", "Pillow>=10.3,<13"), ("matplotlib", "matplotlib"), ("tqdm", "tqdm")]
    if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import json
import pandas as pd
from IPython.display import display, FileLink
from src.methods.dpe_head_reweighting.study import run_study
print("Platform:", PLATFORM, "| Repository:", REPO)
print("Writable workspace:", WORK_DIR)


## 1. Run the complete study

The notebook automatically finds original H2 ZIPs and attached datasets. New backbones use the fixed 100-epoch recipe; each seed trains its own backbone and heads. Within a seed, all 20 weighting candidates share those heads and prediction caches.

Completed backbones/features/heads and frozen selections are reused after identity checks. New backbone runs support epoch-boundary resume. An interrupted head stage restarts its short feature-only training. Legacy incomplete backbone checkpoints lacking the new RNG/history fields cannot be resumed; completed legacy H2 ZIPs remain reusable.

Progress and a compact results ZIP are updated after each completed seed. Outputs contain every planned seed, including failures to improve. Each family and the overall configuration are selected using C, never test WGA.


In [ ]:
if SEARCH_PATHS is None:
    SEARCH_PATHS = [REPO / "outputs/dpe_head_reweighting_runs"]
    if PLATFORM == "kaggle":
        SEARCH_PATHS += [Path("/kaggle/input")]
    elif PLATFORM == "colab":
        SEARCH_PATHS += list(Path("/content").glob("*.zip")) + [Path("/content/uploads")]
    else:
        SEARCH_PATHS += list(working.glob("*.zip"))
STUDY = run_study(WORK_DIR, track=TRACK, seeds=SEEDS, source=SOURCE,
                  search_roots=SEARCH_PATHS, data_dir=DATA_DIR, mode=MODE,
                  device=DEVICE, workers=WORKERS)
print("Study report:", Path(STUDY["output_dir"]) / "report.md")


## 2. Compare all seeds and procedures

`per_seed.csv` shows each seed's C-selected candidate and paired test changes. `summary.json` and `report.md` give means and sample standard deviations across completed seeds. SD is pending when only one seed is complete. A partial study is labeled incomplete until every planned seed finishes.

Inspect these comparisons without changing the grid based on test results. The detailed per-seed reports include all C candidates, exact weights, solver diagnostics, group test accuracies/counts and changed errors.


In [ ]:
summary = STUDY["summary"]
print("Status:", summary["status"], "| Completed seeds:", summary["completed_seeds"])
display(pd.read_csv(Path(STUDY["output_dir"]) / "per_seed.csv"))
display(pd.DataFrame([{
    "procedure": family,
    "mean test accuracy (%)": 100 * values["overall"]["mean"],
    "mean test WGA (%)": 100 * values["wga"]["mean"],
    "WGA sample SD (pp)": None if values["wga"]["sample_std"] is None else 100 * values["wga"]["sample_std"],
    "mean paired WGA change (pp)": values["wga_delta_pp"]["mean"],
    "paired change sample SD (pp)": values["wga_delta_pp"]["sample_std"],
} for family, values in summary["results"].items()]))


## 3. Download the combined results

The ZIP contains the study plan/progress, per-seed and aggregate reports, every weighting comparison, and original H2 prediction bundles for the completed seeds. You do not need to move ZIPs between notebooks. Backbone training/resume checkpoints, raw features and dataset images remain separately in WORK_DIR and are not included in this compact download.

Kaggle provides the file link and saved notebook outputs. Colab also starts a download. If you interrupt the study after a seed completes, you can run this cell separately to download the latest partial results ZIP; rerun the study cell to continue.


In [ ]:
if "STUDY" in globals():
    archive = Path(STUDY["archive"])
else:
    partial = sorted((WORK_DIR / "studies").glob("*.zip"), key=lambda p: p.stat().st_mtime)
    if not partial:
        raise FileNotFoundError("No seed has completed yet. Resume the study cell; backbone checkpoints remain in WORK_DIR.")
    archive = partial[-1]
print("Study archive:", archive)
previous = Path.cwd()
try:
    os.chdir(working)
    display(FileLink(os.path.relpath(archive, working), result_html_prefix="Download results: "))
finally:
    os.chdir(previous)
if PLATFORM == "colab":
    from google.colab import files
    try:
        files.download(str(archive))
    except Exception as error:
        print("Automatic download could not start. Use the link above or the Colab Files panel:", error)
